# VA-TE fusion refit — honest train/val/test protocol

Answers reviewer comments **R1-1, R1-2, R1-6, R2-1** on Electronics 4535789.

**Nothing is retrained.** The projection head is loaded frozen and used for inference only.
The one thing fitted here is the gradient-boosting fusion classifier, on the **training split alone**.

Runtime → Change runtime type → **T4 GPU**. Expect roughly 20–35 minutes end to end.


## 1. Environment

In [ ]:
!nvidia-smi -L || echo "no GPU — set Runtime > Change runtime type > T4 GPU"
!pip -q install rapidfuzz transformers sentencepiece protobuf 2>&1 | tail -2
import torch; print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 2. Data

Cloned from the public repo — no upload needed. The parquet splits live in `data/processed/`.

In [ ]:
![ -d itau-group2 ] || git clone -q https://github.com/sashajh2/itau-group2.git
!ls -la itau-group2/data/processed/*.parquet | head

## 3. Checkpoint

Upload `best_model_siglip_pair.pt` (~2.6 MB). It is in `~/Downloads/` on your Mac.

Swap to the Drive cell below if you would rather mount.

In [ ]:
from google.colab import files
# Upload BOTH files together:
#   best_model_siglip_pair.pt   (~2.6 MB, in ~/Downloads on your Mac)
#   vate_refit.py               (the driver script)
up = files.upload()
print(sorted(up))

In [ ]:
# Alternative: mount Drive instead of uploading.
# from google.colab import drive; drive.mount('/content/drive')
# CKPT_PATH = '/content/drive/MyDrive/path/to/best_model_siglip_pair.pt'

## 4. Run

The script stops hard if VA-TE test ROC-AUC does not reproduce **0.9450**. That gate
matters: if the checkpoint or the encode path is wrong, every downstream number is
meaningless, so it refuses to continue rather than producing plausible garbage.

Embedding is resumable — if the session drops, rerun this cell and it picks up from the cache.

In [ ]:
%run vate_refit.py

## 5. Collect

`corrected_tables.md` has LaTeX rows ready to paste into `template.tex`.

In [ ]:
from google.colab import files
print(open('vate_out/corrected_tables.md').read())
files.download('vate_out/results.json')
files.download('vate_out/corrected_tables.md')

## What to send back

1. The **projection head** line the script prints in bold — settles whether Table 12 says 512 or 768.
2. Whether the **gate passed**.
3. `corrected_tables.md`.
4. The **bootstrap** block — specifically whether `d_vs_vate` still excludes zero. That is
   the number that decides whether the paper's central claim survives as written.
